In [2]:
import os
import time
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt
from tensorflow.keras.applications import ResNet50
from tensorflow.keras import layers, models, metrics

print("TensorFlow Version:", tf.__version__)
print("GPU Available for Training:", tf.config.list_physical_devices('GPU'))

TensorFlow Version: 2.21.0
GPU Available for Training: []


In [3]:
PROCESSED_DATA_PATH = "../processed_data"

print("🔄 Loading 5-class processed datasets from disk...")
ds_train = tf.data.Dataset.load(os.path.join(PROCESSED_DATA_PATH, 'train'))
ds_val = tf.data.Dataset.load(os.path.join(PROCESSED_DATA_PATH, 'val'))
ds_test = tf.data.Dataset.load(os.path.join(PROCESSED_DATA_PATH, 'test'))

print("✅ Data successfully injected into runtime pipeline.")

🔄 Loading 5-class processed datasets from disk...
✅ Data successfully injected into runtime pipeline.


In [4]:
# 1. Load the pre-trained ResNet50 backbone model without its top classification layer
base_model = ResNet50(weights='imagenet', include_top=False, input_shape=(224, 224, 3))

# 2. Freeze the base layers (Transfer Learning strategy)
base_model.trainable = False

# 3. Match your 5 plant health classes
NUM_CLASSES = 5 

# 4. Append custom evaluation layers on top
model = models.Sequential([
    base_model,
    layers.GlobalAveragePooling2D(),
    layers.Dropout(0.2), # Dropout layer to prevent overfitting during training
    layers.Dense(NUM_CLASSES, activation='softmax')
])

# Compile tracking Accuracy and Top-1 Match (as proxy for mAP)
model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=[
        'accuracy',
        metrics.SparseTopKCategoricalAccuracy(k=1, name='mAP')
    ]
)

model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ resnet50 (Functional)                │ (None, 7, 7, 2048)          │      23,587,712 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ global_average_pooling2d             │ (None, 2048)                │               0 │
│ (GlobalAveragePooling2D)             │                             │                 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout (Dropout)                    │ (None, 2048)                │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense (Dense)                        │ (None, 5)                   │          10,245 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 23,597,957 (90.02 MB)

 Trainable params: 10,245 (40.02 KB)

 Non-trainable params: 23,587,712 (89.98 MB)

In [5]:
EPOCHS = 50

print(f"🚀 Launching ResNet50 training pipeline for {EPOCHS} epochs...")
start_time = time.time()

history = model.fit(
    ds_train,
    validation_data=ds_val,
    epochs=EPOCHS
)

end_time = time.time()
resnet_total_time = end_time - start_time

print("\n--- 🏁 RESNET50 TRAINING COMPLETE ---")
print(f"Total Execution Time: {resnet_total_time:.2f} seconds")

🚀 Launching ResNet50 training pipeline for 50 epochs...
Epoch 1/50
134/134 ━━━━━━━━━━━━━━━━━━━━ 276s 2s/step - accuracy: 0.3526 - loss: 1.4179 - mAP: 0.3526 - val_accuracy: 0.4449 - val_loss: 1.3202 - val_mAP: 0.4449
Epoch 2/50
134/134 ━━━━━━━━━━━━━━━━━━━━ 409s 3s/step - accuracy: 0.4142 - loss: 1.3097 - mAP: 0.4142 - val_accuracy: 0.5037 - val_loss: 1.2579 - val_mAP: 0.5037
Epoch 3/50
134/134 ━━━━━━━━━━━━━━━━━━━━ 429s 3s/step - accuracy: 0.4609 - loss: 1.2585 - mAP: 0.4609 - val_accuracy: 0.5423 - val_loss: 1.2101 - val_mAP: 0.5423
Epoch 4/50
134/134 ━━━━━━━━━━━━━━━━━━━━ 439s 3s/step - accuracy: 0.4862 - loss: 1.2172 - mAP: 0.4862 - val_accuracy: 0.5588 - val_loss: 1.1772 - val_mAP: 0.5588
Epoch 5/50
134/134 ━━━━━━━━━━━━━━━━━━━━ 460s 3s/step - accuracy: 0.5091 - loss: 1.1870 - mAP: 0.5091 - val_accuracy: 0.5827 - val_loss: 1.1475 - val_mAP: 0.5827
Epoch 6/50
134/134 ━━━━━━━━━━━━━━━━━━━━ 622s 5s/step - accuracy: 0.5279 - loss: 1.1648 - mAP: 0.5279 - val_accuracy: 0.6103 - val_loss: 1.1

In [6]:
import os
import csv

# Ensure the results directory exists
os.makedirs('../results', exist_ok=True)

# 1. Capture the history dictionary from your training run
history_dict = history.history
csv_path = '../results/resnet50_history.csv'

# 2. Extract metrics and write them row-by-row to CSV
with open(csv_path, mode='w', newline='') as f:
    writer = csv.writer(f)
    keys = list(history_dict.keys())
    writer.writerow(keys) # Header row
    
    num_epochs = len(history_dict[keys[0]])
    for i in range(num_epochs):
        row = [history_dict[k][i] for k in keys]
        writer.writerow(row)

# 3. Save the model configuration using the modern native extension
model.save('../results/resnet50_model.keras')

# 4. Log the execution time so it's cached for references
with open('../results/resnet50_time.txt', 'w') as f:
    f.write(str(resnet_total_time))

print("💾 SUCCESS! ResNet50 history exported to CSV and model architecture saved.")

💾 SUCCESS! ResNet50 history exported to CSV and model architecture saved.
